### Step 0: connect Google Drive (only if you are using Colab)

This notebook runs on Google Colab, which runs the code in a temporary virtual machine in the cloud. The machine is deleted when it is idle for a while or reaches its maximum lifetime. It starts without our files, so the code cannot see the files in Google Drive, even though the notebook itself is saved there. Those files are the pretrained model `sft/gpt.pt`, the codebook `sft/meta.pkl`, `model.py` and the training data. Mounting Google Drive lets the notebook's code read and write those files ([Google, n.d.](https://research.google.com/colaboratory/faq.html)).


The cell below:

1. mounts Google Drive, so its files appear under `/content/drive/MyDrive/`;
2. moves into the `dpo/` folder, because the code loads files by paths relative to it (`../sft/gpt.pt`, `../model.py`);
3. prints the GPU (`nvidia-smi`) and lists the files, to check that both are there before anything runs.

Mounting also means checkpoints and results are saved to Drive. Files saved only on the Colab machine are deleted when the session ends.

**Skip this cell if you are not running this from Colab.**

 from google.colab import drive
 drive.mount('/content/drive')
 %cd /content/drive/MyDrive/SC3000/NanoGPT-Math/dpo
 !nvidia-smi
 !ls -la ../sft . baseline

### Step 1: Install necessary packages

`%pip` installs into the environment of the running kernel. `!pip` runs whichever `pip` comes first on the shell's PATH, which can belong to a different Python. Only `torch`, `numpy`, `matplotlib` and `tqdm` are used below; the other packages are kept from the template.

In [1]:
%pip install matplotlib
%pip install torch numpy transformers datasets tiktoken wandb tqdm

/Users/jeromeke/Projects/SC3000-A1-experiments/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/Users/jeromeke/Projects/SC3000-A1-experiments/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


### Step 2: Package imports and configuration

Changes from the template:
- **Device.** The template only checks for CUDA. We also use Apple's GPU (`mps`) when there is one, and fall back to the CPU otherwise.
- **`CUDA_VISIBLE_DEVICES = "1"` removed.** That line hides every GPU except the second one. On a machine with a single GPU, PyTorch would then see no GPU at all and silently train on the CPU.
- **`max_length = 80` instead of 64.** The longest positives, such as `x*100=10000,x=? The answer is 100 because 10000/100 equals to 100.`, are 70 characters long with the 4 newlines that `get_batches` adds. `pad_or_truncate` keeps the *last* `max_length` characters, so with 64 the start of these samples, the question itself, would be cut off. A few negatives are longer than 80, when the pretrained model keeps on talking; for them `pad_or_truncate` cuts off the start.
- **Learning rate `1e-3` instead of `1e-4`, and 12 epochs instead of 5.** In a one-epoch trial run with the template's `1e-4`, the model learned the sentence pattern but answered only 1.7% of the test questions correctly: it still guessed the answer digits. Learning the arithmetic itself is the slow part, so we raised the learning rate and trained for longer. With these settings the accuracy also stays low for the first few epochs and then jumps; Step 7 prints it after every epoch. We did not test whether `1e-4` with 12 epochs would do as well.
- `re` and `Counter` are imported for Task 1 and Step 8.

The other settings are the template's: `beta = 0.5`, batch size 64, and the sampling settings for `gpt.generate` (temperature 0.8, top_k 200, at most 200 new characters).

The tokenizer works on single characters: `meta.pkl` maps each of the 74 characters to an id. The newline character has id 0, which is also used for padding, and `.` has id 7. `gpt.generate` in `model.py` stops at either of these two.

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
from tqdm import tqdm
import time
import json
import re
from collections import Counter
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print("device: ", device)
base_lr = 1e-3
epochs = 12
batch_size = 64
max_length = 80
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

device:  mps


### Step 3: Define helper functions

These three functions come with the template and are used unchanged:
- `compute_logprob(input_ids)` runs the model on each sequence and returns the **average log-probability per character**, leaving out padding (id 0). For a batch of `B` sequences it returns a tensor of shape `B`.
- `pad_or_truncate(seq, max_length)` pads a sequence with id 0 up to `max_length`. A longer sequence keeps only its last `max_length` ids.
- `get_batches(lines, batch_size)` shuffles the pairs, appends 4 newlines to each negative and positive, encodes and pads them, and yields `(neg_tensor, pos_tensor)`, each of shape `batch_size × max_length`. The last incomplete batch is dropped.

Because the newline is id 0, the 4 newlines appended in `get_batches` are treated as padding and never scored. The model therefore learns to end a reply with `.`, which is also where `gpt.generate` stops.

In [2]:
def compute_logprob(model, input_ids):   # pass the model in, so it scores whichever model we're training
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = model(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

This loads `sft/gpt.pt`: 6 layers, 6 attention heads, 348-dimensional embeddings, a context of 256 characters and a 74-character vocabulary, 8.8M parameters in total. The `_orig_mod.` prefix that `torch.compile` adds to the weight names is removed before loading. The model is left in training mode, so dropout (p = 0.2) is on.

In [3]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Task 1: Build positive-negative data pairs

This cell builds 100,202 (negative, positive) pairs and saves them to `./pos_neg_pairs.json` (that is, `dpo/pos_neg_pairs.json`), in the same format as the sample file. Step 5 loads this file.

**Question types.** There are 12 types: the four operations asked directly (`a+b=?`, `a-b=?`, `a*b=?`, `a/b=?`), and each operation as an equation with `x` on the left (`x+a=b,x=?`) or on the right (`a+x=b,x=?`). The format copies the Step 8 test set exactly, with no space after the comma, because the model reads one character at a time.

**Numbers.** For each type we draw two numbers from 0–100 and work out the third from them, so the answer is always a whole number. For example, for `a/b=?` we draw `b` and the answer, then set `a = b*answer`. A pair is skipped if it would divide by 0, give a negative answer, or give an equation with more than one solution (such as `0*x=0`). For each type we list every allowed pair and shuffle them. The first 50 are kept aside as **test questions**: these exact questions are never trained on, and Steps 7 and 8 measure accuracy on them. The same arithmetic fact can still appear in training in another form, for example `14*24=?` for the test question `24*14=?`. The next 9000 become training pairs. The two subtraction types only have about 5150 allowed pairs each, so the total is 100,202 training pairs.

**Positive.** The same wording as the sample file: `79-7=? The answer is 72 because 79-7 equals 72.` for plain calculations, and `x+55=95,x=? The answer is 40 because 95-55 equals to 40.` for equations, where the working after "because" is the reverse operation.

**Negative.** The pretrained model's own reply to the question, generated with `gpt.generate` and the template's sampling settings (temperature 0.8, top_k 200); it stops at `.` or a newline. One question takes about 0.1 s, so all of them took 190 minutes here.

**Output.** The example pairs show what the pretrained model does with arithmetic: it refuses, with "Sorry, I don't know." or variants such as the misspelled "Sory, I don't know." and "I don't know."

**Checking answers.** `reply_is_correct` reads the number after "The answer is" in the model's reply, puts it back into the question and checks the arithmetic: for `x+55=95,x=?` and the answer 40, it checks that 40+55 equals 95. `test_accuracy` uses it to count how many of the 600 test questions the model gets right.

**Two cells.** The first cell builds the questions and positives and defines all functions; it runs in a second. The second cell calls `main`, which generates the negatives and writes `pos_neg_pairs.json`; it is the slow one.

**Reproducibility.** The seed makes the questions and positives the same on every machine. The negatives also depend on the device's random number generator, so running this cell on another device gives slightly different negatives and overwrites `pos_neg_pairs.json`.

In [8]:
random.seed(0)
torch.manual_seed(0)
numbers = range(0, 101)
max_per_type = 9000            # training questions per type
test_per_type = 50             # questions per type kept out of training, used as test questions
data_path = "./pos_neg_pairs.json"

# The 12 question types. Each function takes two numbers drawn from 0..100 and returns
# (question, answer, working), where `working` is the calculation written after "because",
# or None when the pair is not allowed.
def add(a, b):    return f"{a}+{b}=?", a + b, f"{a}+{b}"
def sub(a, b):    return (f"{a}-{b}=?", a - b, f"{a}-{b}") if b <= a else None
def mul(a, b):    return f"{a}*{b}=?", a * b, f"{a}*{b}"
def div(b, q):    return (f"{b*q}/{b}=?", q, f"{b*q}/{b}") if b > 0 else None

def x_add(x, a):  return f"x+{a}={x+a},x=?", x, f"{x+a}-{a}"
def add_x(x, a):  return f"{a}+x={a+x},x=?", x, f"{a+x}-{a}"
def x_sub(a, b):  return f"x-{a}={b},x=?", a + b, f"{b}+{a}"
def sub_x(a, x):  return (f"{a}-x={a-x},x=?", x, f"{a}-{a-x}") if x <= a else None
def x_mul(x, a):  return (f"x*{a}={x*a},x=?", x, f"{x*a}/{a}") if a > 0 else None
def mul_x(x, a):  return (f"{a}*x={a*x},x=?", x, f"{a*x}/{a}") if a > 0 else None
def x_div(a, b):  return (f"x/{a}={b},x=?", a * b, f"{b}*{a}") if a > 0 else None
def div_x(x, b):  return (f"{x*b}/x={b},x=?", x, f"{x*b}/{b}") if x > 0 and b > 0 else None


question_types = [add, sub, mul, div, x_add, add_x, x_sub, sub_x, x_mul, mul_x, x_div, div_x]

def build_questions(make):
    """Collect every allowed question of one type, in random order."""
    items = []
    for p in numbers:
        for q in numbers:
            item = make(p, q)
            if item is not None:              # None: this pair of numbers is not allowed
                items.append(item)
    random.shuffle(items)
    return items


def make_positive(question, answer, working):
    """Write the positive reply for one question: the question, the answer and its working."""
    # The sample file dpo/pos_neg_pairs.json (Figure 4) writes "equals" for plain calculations
    # and "equals to" for equations with x, so we use the same wording.
    if "x" in question:
        link = "equals to"
    else:
        link = "equals"
    return f"{question} The answer is {answer} because {working} {link} {answer}."

unused_questions = {}   # leftover mul / x_div questions (product set), not in train or test
questions = []
positives = []
test_items = []                # (question, answer, working), not used for training
for make in question_types:
    items = build_questions(make)
    test_items += items[:test_per_type]
    for question, answer, working in items[test_per_type:test_per_type + max_per_type]:
        questions.append(question)
        positives.append(make_positive(question, answer, working))

    # keep whatever's left after test and train, only for mul and x_div
    if make is mul or make is x_div:
        unused_questions[make.__name__] = items[test_per_type + max_per_type:]

print("Number of training questions:", len(questions))
print("Number of test questions:", len(test_items))
print("Example positive:", positives[0])

# quick look at the leftover sets
for name, rest in unused_questions.items():
    print(name, "leftovers:", len(rest))
    print("  first:", rest[0][0])
    print("  last: ", rest[-1][0])

def generate_reply(question):
    """The model's reply to one question, sampled with gpt.generate; the returned text starts with the question."""
    x = torch.tensor([encode(question)], dtype=torch.long, device=device)
    y, _ = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(y[0].tolist())

def main(questions, positives, path):
    """Let the pretrained model answer every question, then save the (negative, positive) pairs to `path`."""
    start = time.time()
    gpt.eval()                    # no dropout while sampling
    negatives = []
    for question in tqdm(questions):
        negatives.append(generate_reply(question))
    gpt.train()                   # back to training mode for Step 7
    print(f"Generated {len(negatives)} negatives in {(time.time() - start) / 60:.1f} minutes")

    pairs = []
    for negative, positive in zip(negatives, positives):
        pairs.append({"negative": negative, "positive": positive})
    with open(path, "w", encoding="utf-8") as f:
        json.dump(pairs, f, ensure_ascii=False, indent=1)
    print(f"Saved {len(pairs)} pairs to {path}")
    print("Some example pairs:")
    for pair in random.sample(pairs, 5):
        print(pair)

def reply_is_correct(question, text):
    """Put the number after "The answer is" in the model's text back into the question and check the arithmetic."""
    match = re.search(r"The answer is (\d+)", text[len(question):], re.IGNORECASE)
    if match is None:
        return False
    answer = int(match.group(1))                         # int, so a reply like "06" becomes 6
    if "x" in question:                                  # "x+55=95,x=?" -> 40+55 == 95
        left, right = question.split(",")[0].replace("x", str(answer)).split("=")
        try:
            return eval(left) == eval(right)
        except ZeroDivisionError:                        # answering 0 to "84/x=7,x=?"
            return False
    return eval(question[:-2]) == answer                 # "79-7=?" -> 79-7 == 72

test_questions = [question for question, answer, working in test_items]

def test_accuracy():
    """Answer the test questions with the current model; returns (share correct, {question: reply})."""
    gpt.eval()
    reply_of = {}
    for question in test_questions:
        reply_of[question] = generate_reply(question)
    gpt.train()
    correct = 0
    for question in test_questions:
        if reply_is_correct(question, reply_of[question]):
            correct += 1
    return correct / len(test_questions), reply_of

def question_type(question):
    return re.sub(r"\d+", "n", question)          # "x+55=95,x=?" -> "x+n=n,x=?"

Number of training questions: 100202
Number of test questions: 600
Example positive: 79+7=? The answer is 86 because 79+7 equals 86.
mul leftovers: 1151
  first: 83*56=?
  last:  61*1=?
x_div leftovers: 1050
  first: x/11=87,x=?
  last:  x/52=73,x=?


In [6]:
# Generate the negatives with the pretrained model and save all pairs (about 3 hours on an Apple M2 Pro).
# To test the submitted dpo.pt without rebuilding the data, skip this cell (see "How to run" at the top).
# main(questions, positives, data_path)

### Step 5: Load Data

We load the file written by Task 1. The template comment mentions `./data/pos_neg_pairs.json`, but there is no `data/` folder; the file the manual asks us to submit is `dpo/pos_neg_pairs.json`, which is `data_path` here. The list is called `lines` because `get_batches` in Step 7 expects that name.

In [7]:
# Load the pairs built in Task 1
with open(data_path, encoding="utf-8") as f:
    lines = json.load(f)
print("Number of pairs:", len(lines))
print("Example pair:", lines[0])

Number of pairs: 100202
Example pair: {'negative': "79+7=? Sorry, I don't know.", 'positive': '79+7=? The answer is 86 because 79+7 equals 86.'}


### Step 6: Build the optimizer and scheduler

- **AdamW**, as the template recommends, with the learning rate `base_lr = 1e-3` from Step 2 and a small weight decay (0.01).
- **Cosine schedule.** The learning rate starts at `1e-3` and falls along a cosine curve to `1e-4` at the last step. Large steps early on get the model through the slow first phase of learning arithmetic; small steps at the end let it settle on exact digits.
- With 100,202 pairs and a batch size of 64 there are 1,565 steps per epoch, 18,780 in total.

In [8]:
# recommend to use the AdamW optimizer
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr, weight_decay=0.01)

# The learning rate goes from base_lr down to base_lr/10 along a cosine curve over the whole training run
steps_per_epoch = len(lines) // batch_size      # get_batches drops the last incomplete batch
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * steps_per_epoch, eta_min=base_lr / 10)
print("Training steps:", epochs * steps_per_epoch)

Training steps: 18780


## My greedy evaluator

In [10]:
# greedy evaluator: top_k=1 means the model always picks its most likely char
# so the same model gives the same replies every run
def greedy_eval(model, items):
    model.eval()   # turn off dropout
    t0 = time.time()
    counts = {}    # question type -> counter with total / correct / wrong / format_fail
    replies = {}   # question -> full reply

    with torch.no_grad():
        for q, ans, work in items:
            # same as generate_reply, but on the model passed in and with top_k=1
            x = torch.tensor([encode(q)], dtype=torch.long, device=device)
            y, _ = model.generate(x, max_new_tokens, temperature=temperature, top_k=1)
            reply = decode(y[0].tolist())
            replies[q] = reply

            qtype = question_type(q)
            if qtype not in counts:
                counts[qtype] = Counter()
            counts[qtype]["total"] += 1
            if reply_is_correct(q, reply):   # reusing checker from Task 1
                counts[qtype]["correct"] += 1
            elif re.search(r"The answer is (\d+)", reply[len(q):], re.IGNORECASE):   # gave a number, just the wrong one
                counts[qtype]["wrong"] += 1
            else:   # no number at all, e.g. "Sorry, I don't know."
                counts[qtype]["format_fail"] += 1

    model.train()   # back to train mode, since training will call this between epochs

    total = Counter()
    for c in counts.values():
        total.update(c)
    counts["All"] = total
    print(f"took {time.time() - t0:.1f}s")

    print(f"{'type':<12}{'total':>7}{'correct':>9}{'wrong num':>11}{'format fail':>13}{'% correct':>11}")
    for qtype, c in counts.items():
        pct = 100 * c["correct"] / c["total"]
        print(f"{qtype:<12}{c['total']:>7}{c['correct']:>9}{c['wrong']:>11}{c['format_fail']:>13}{pct:>10.1f}%")
    return counts, replies

In [10]:
# checking the same model twice must give identical tables
counts_a, replies_a = greedy_eval(gpt, test_items)
counts_b, replies_b = greedy_eval(gpt, test_items)

took 35.8s
type          total  correct  wrong num  format fail  % correct
n+n=?            50        0          0           50       0.0%
n-n=?            50        0          0           50       0.0%
n*n=?            50        0          0           50       0.0%
n/n=?            50        0          0           50       0.0%
x+n=n,x=?        50        0          0           50       0.0%
n+x=n,x=?        50        0          0           50       0.0%
x-n=n,x=?        50        0          0           50       0.0%
n-x=n,x=?        50        0          0           50       0.0%
x*n=n,x=?        50        0          0           50       0.0%
n*x=n,x=?        50        0          0           50       0.0%
x/n=n,x=?        50        0          0           50       0.0%
n/x=n,x=?        50        0          0           50       0.0%
All             600        0          0          600       0.0%
took 34.9s
type          total  correct  wrong num  format fail  % correct
n+n=?            5

In [11]:
# Run 0 (pretrained) on the product set
product_set = unused_questions["mul"] + unused_questions["x_div"]   # 1151 + 1050 = 2201
run0_prod_counts, run0_prod_replies = greedy_eval(gpt, product_set)

# load the team's model into its own variable so gpt stays the pretrained one
checkpoint = torch.load("baseline/dpo.pt", map_location=device)
print(checkpoint.keys())   # just to see what's saved in there

team_model = GPT(GPTConfig(**checkpoint["model_args"]))
state_dict = checkpoint["model_state_dict"]
# strip the _orig_mod. prefix from torch.compile, same as Step 4 (does nothing if it's not there)
for k in list(state_dict.keys()):
    if k.startswith("_orig_mod."):
        state_dict[k[len("_orig_mod."):]] = state_dict.pop(k)
team_model.load_state_dict(state_dict)
team_model.to(device)

# Run 1 (team's model): test set first, then the product set
run1_test_counts, run1_test_replies = greedy_eval(team_model, test_items)
run1_prod_counts, run1_prod_replies = greedy_eval(team_model, product_set)

took 134.4s
type          total  correct  wrong num  format fail  % correct
n*n=?          1151        0          0         1151       0.0%
x/n=n,x=?      1050        0          0         1050       0.0%
All            2201        0          0         2201       0.0%
dict_keys(['model_state_dict', 'model_args'])
took 98.6s
type          total  correct  wrong num  format fail  % correct
n+n=?            50       50          0            0     100.0%
n-n=?            50       50          0            0     100.0%
n*n=?            50       17         33            0      34.0%
n/n=?            50       39         11            0      78.0%
x+n=n,x=?        50       50          0            0     100.0%
n+x=n,x=?        50       50          0            0     100.0%
x-n=n,x=?        50       50          0            0     100.0%
n-x=n,x=?        50       50          0            0     100.0%
x*n=n,x=?        50       46          4            0      92.0%
n*x=n,x=?        50       43       

In [12]:
# first 10 wrong-number replies on the product set, in list order so it's the same 10 every time
wrong = []
for q, ans, work in product_set:
    reply = run1_prod_replies[q]
    # same test as in greedy_eval: gave a number, but not the right one
    if not reply_is_correct(q, reply) and re.search(r"The answer is (\d+)", reply[len(q):]):
        wrong.append((q, reply, ans))
    if len(wrong) == 10:
        break

for q, reply, ans in wrong:
    print("Q:    ", q)
    print("reply:", reply)
    print("right:", ans)
    print()

Q:     83*56=?
reply: 83*56=? The answer is 4448 because 83*56 equals 4448.
right: 4648

Q:     69*14=?
reply: 69*14=? The answer is 946 because 69*14 equals 946.
right: 966

Q:     34*72=?
reply: 34*72=? The answer is 2388 because 34*72 equals 2388.
right: 2448

Q:     61*57=?
reply: 61*57=? The answer is 3037 because 61*57 equals 3037.
right: 3477

Q:     59*18=?
reply: 59*18=? The answer is 1042 because 59*18 equals 1042.
right: 1062

Q:     83*98=?
reply: 83*98=? The answer is 8114 because 83*98 equals 8114.
right: 8134

Q:     73*75=?
reply: 73*75=? The answer is 5035 because 73*75 equals 5035.
right: 5475

Q:     38*11=?
reply: 38*11=? The answer is 498 because 38*11 equals 498.
right: 418

Q:     45*28=?
reply: 45*28=? The answer is 1240 because 45*28 equals 1240.
right: 1260

Q:     94*62=?
reply: 94*62=? The answer is 5788 because 94*62 equals 5788.
right: 5828



In [13]:
replies_a == replies_b

True

### Step 7: Begin training

**Loss.** We use the loss suggested in the template:

$$\text{loss} = -\log\sigma\!\left(\frac{\log p(\text{pos}) - \log p(\text{neg})}{\beta}\right) - 0.1\,\log p(\text{pos})$$

The first term is the DPO preference loss, and the second is an SFT (supervised fine-tuning) loss on the positive reply. Here $\log p(\cdot)$ is the average log-probability per character of a whole sequence (question and reply), computed by `compute_logprob` from Step 3, and $\beta = 0.5$.

**Test after every epoch.** The loss only shows how closely the model copies the training replies, not whether it can really calculate. So after every epoch the model answers the 600 test questions from Task 1, which it never trains on, and we print how many it gets right, along with one example reply for six of the question types. This shows when the model moves from copying the sentence pattern to actually calculating, and whether more epochs still help.

**Other details.** Gradients are clipped to a norm of 1.0, so a single unusual batch cannot move the model too far. The progress bar shows the current loss. The checkpoint `dpo.pt` is saved after every epoch, as in the template.

In [23]:
def train_run(name, epochs, pairs):
    # same seeds every run, so runs only differ in what I changed
    random.seed(0)
    torch.manual_seed(0)
    pairs = list(pairs)   # get_batches shuffles in place, so shuffle a copy and leave lines alone

    # fresh pretrained model for each run (same as Step 4, but called model)
    ckpt = torch.load("../sft/gpt.pt", map_location=device)
    gptconf = GPTConfig(**ckpt['model_args'])
    model = GPT(gptconf)
    state_dict = ckpt['model']
    unwanted_prefix = '_orig_mod.'
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    model.load_state_dict(state_dict)
    model.to(device).train()

    # new optimizer + scheduler per run (same as Step 6), so the cosine decays over this run's epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=0.01)
    steps_per_epoch = len(pairs) // batch_size
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * steps_per_epoch, eta_min=base_lr / 10)

    loss_history = []
    margin_history = []        # pos log-prob minus neg log-prob
    pref_acc_history = []      # share of the batch where pos beats neg
    pos_logprob_history = []   # avg log-prob of the positive (for Experiment D)
    start = time.time()
    total_steps = len(pairs) // batch_size
    for epoch in range(epochs):
        pbar = tqdm(get_batches(pairs, batch_size), total=total_steps)
        for step, (neg_tensor, pos_tensor) in enumerate(pbar):
            neg_logprob = compute_logprob(model, neg_tensor)
            pos_logprob = compute_logprob(model, pos_tensor)
            # DPO part: make the positive more likely than the negative.
            # SFT part (- 0.1 * pos_logprob): also raise the probability of the positive directly.
            loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            loss_history.append(loss.item())
            margin_history.append((pos_logprob - neg_logprob).mean().item())
            pref_acc_history.append((pos_logprob > neg_logprob).float().mean().item())
            pos_logprob_history.append(pos_logprob.mean().item())
            pbar.set_description(f"epoch {epoch + 1}/{epochs}, loss {loss.item():.3f}")
        ckpt_path = f"runs/{name}.pt"   # own file per run, never overwrites dpo.pt
        torch.save({
            "model_state_dict": model.state_dict(),
            "model_args": ckpt['model_args'],
        }, ckpt_path)
        epoch_loss = sum(loss_history[-total_steps:]) / total_steps
        print(f"Epoch {epoch + 1}: average loss {epoch_loss:.4f}")

    minutes = (time.time() - start) / 60
    print(f"Training took {minutes:.1f} minutes")

    # score this run greedily, same as Run 0 / Run 1
    test_counts, _ = greedy_eval(model, test_items)
    prod_counts, _ = greedy_eval(model, product_set)

    # save everything, so a restart doesn't lose an hour of results
    results = {
        "name": name,
        "epochs": epochs,
        "minutes": minutes,
        "loss": loss_history,
        "margin": margin_history,
        "pref_acc": pref_acc_history,
        "pos_logprob": pos_logprob_history,
        "test_counts": test_counts,
        "prod_counts": prod_counts,
    }
    with open(f"runs/{name}.json", "w") as f:
        json.dump(results, f)
    return results

In [24]:
run2 = train_run("run2_baseline", 6, lines)

epoch 1/6, loss 0.032: 100%|█████████████████████████████████████████████████████████| 1565/1565 [09:56<00:00,  2.62it/s]


Epoch 1: average loss 0.0591


epoch 2/6, loss 0.027: 100%|█████████████████████████████████████████████████████████| 1565/1565 [09:31<00:00,  2.74it/s]


Epoch 2: average loss 0.0292


epoch 3/6, loss 0.023: 100%|█████████████████████████████████████████████████████████| 1565/1565 [09:26<00:00,  2.76it/s]


Epoch 3: average loss 0.0249


... (2 outputs left out)

epoch 5/6, loss 0.022: 100%|█████████████████████████████████████████████████████████| 1565/1565 [09:39<00:00,  2.70it/s]


Epoch 5: average loss 0.0221


epoch 6/6, loss 0.022: 100%|█████████████████████████████████████████████████████████| 1565/1565 [09:36<00:00,  2.72it/s]


Epoch 6: average loss 0.0217
Training took 58.1 minutes
took 93.3s
type          total  correct  wrong num  format fail  % correct
n+n=?            50       50          0            0     100.0%
n-n=?            50       47          3            0      94.0%
n*n=?            50        9         41            0      18.0%
n/n=?            50       35         15            0      70.0%
x+n=n,x=?        50       50          0            0     100.0%
n+x=n,x=?        50       49          1            0      98.0%
x-n=n,x=?        50       50          0            0     100.0%
n-x=n,x=?        50       43          7            0      86.0%
x*n=n,x=?        50       37         13            0      74.0%
n*x=n,x=?        50       38         12            0      76.0%
x/n=n,x=?        50       12         38            0      24.0%
n/x=n,x=?        50       39         11            0      78.0%
All             600      459        141            0      76.5%
took 1456.8s
type          total  cor

### Experiment A: hard negatives

For the 9,000 `a*b=?` and 9,000 `x/a=b,x=?` training pairs, the negative "Sorry, I don't know." is swapped for the same sentence as the positive with one wrong digit in the answer. A refusal is easy to rank below the right answer, but a hard negative only differs in the digits, so the model has to get the digits right to prefer the positive.

In [25]:
# swap the refusals for a*b and x/a with almost right answers
random.seed(0)   # same wrong answers every time this cell runs

def wrong_answer(n):
    # change one digit: a middle one if 3+ digits, otherwise the last one
    # first digit never changes -> no leading zero, same length, always wrong
    digits = list(str(n))
    if len(digits) >= 3:
        i = random.randrange(1, len(digits) - 1)
    else:
        i = len(digits) - 1
    digits[i] = random.choice("0123456789".replace(digits[i], ""))   # anything but the old digit
    return int("".join(digits))

hard_lines = []
changed = 0
examples = []   # first few changed pairs, to print at the end
for p in lines:
    pos = p["positive"]
    question = pos[:pos.index("?") + 1]   # same as the overlap check
    qtype = question_type(question)
    if qtype != "n*n=?" and qtype != "x/n=n,x=?":
        hard_lines.append(p)   # every other type stays as it is
        continue

    a, b = [int(x) for x in re.findall(r"\d+", question)]   # 83*56=? -> 83, 56 / x/11=87,x=? -> 11, 87
    if qtype == "n*n=?":
        q, ans, working = mul(a, b)
    else:
        q, ans, working = x_div(a, b)
    assert make_positive(q, ans, working) == pos   # sanity check: rebuilt item matches the real positive

    # build the negative with the same template instead of .replace(), so the question can't get changed
    neg = make_positive(q, wrong_answer(ans), working)
    new_pair = {"positive": pos, "negative": neg}   # new dict, so lines (and Run 2's data) isn't touched
    hard_lines.append(new_pair)
    changed += 1
    if len(examples) < 5:
        examples.append(new_pair)

print("rows changed:", changed)
print("total rows:", len(hard_lines))
for p in examples:
    print("pos:", p["positive"])
    print("neg:", p["negative"])
    print()

rows changed: 18000
total rows: 100202
pos: 57*67=? The answer is 3819 because 57*67 equals 3819.
neg: 57*67=? The answer is 3879 because 57*67 equals 3879.

pos: 29*35=? The answer is 1015 because 29*35 equals 1015.
neg: 29*35=? The answer is 1515 because 29*35 equals 1515.

pos: 97*73=? The answer is 7081 because 97*73 equals 7081.
neg: 97*73=? The answer is 7061 because 97*73 equals 7061.

pos: 27*37=? The answer is 999 because 27*37 equals 999.
neg: 27*37=? The answer is 979 because 27*37 equals 979.

pos: 54*68=? The answer is 3672 because 54*68 equals 3672.
neg: 54*68=? The answer is 3632 because 54*68 equals 3632.



In [26]:
run3 = train_run("run3_hard_negatives", 6, hard_lines)

epoch 1/6, loss 0.134: 100%|█████████████████████████████████████████████████████████| 1565/1565 [09:32<00:00,  2.73it/s]


Epoch 1: average loss 0.2161


epoch 2/6, loss 0.222: 100%|█████████████████████████████████████████████████████████| 1565/1565 [09:51<00:00,  2.64it/s]


Epoch 2: average loss 0.2051


epoch 3/6, loss 0.222: 100%|█████████████████████████████████████████████████████████| 1565/1565 [15:13<00:00,  1.71it/s]


Epoch 3: average loss 0.2041


... (2 outputs left out)

epoch 5/6, loss 0.200: 100%|█████████████████████████████████████████████████████████| 1565/1565 [09:42<00:00,  2.68it/s]


Epoch 5: average loss 0.1943


epoch 6/6, loss 0.184: 100%|█████████████████████████████████████████████████████████| 1565/1565 [09:26<00:00,  2.76it/s]


Epoch 6: average loss 0.1916
Training took 78.1 minutes
took 81.0s
type          total  correct  wrong num  format fail  % correct
n+n=?            50        1         49            0       2.0%
n-n=?            50        2         48            0       4.0%
n*n=?            50        0         50            0       0.0%
n/n=?            50        1         49            0       2.0%
x+n=n,x=?        50        0         50            0       0.0%
n+x=n,x=?        50        2         48            0       4.0%
x-n=n,x=?        50        0         50            0       0.0%
n-x=n,x=?        50        1         49            0       2.0%
x*n=n,x=?        50        0         50            0       0.0%
n*x=n,x=?        50        1         49            0       2.0%
x/n=n,x=?        50        0         50            0       0.0%
n/x=n,x=?        50        0         50            0       0.0%
All             600        8        592            0       1.3%
took 292.2s
type          total  corr

In [12]:
# the reasoning model
checkpoint = torch.load("baseline/reasoning_dpo.pt", map_location=device)
print(checkpoint.keys())   # see what's saved in there

reasoning_model = GPT(GPTConfig(**checkpoint["model_args"]))
state_dict = checkpoint["model_state_dict"]

# strip the _orig_mod. prefix from torch.compile
for k in list(state_dict.keys()):
    if k.startswith("_orig_mod."):
        state_dict[k[len("_orig_mod."):]] = state_dict.pop(k)
reasoning_model.load_state_dict(state_dict)
reasoning_model.to(device)

# score it on both sets and keep the results
reason_test_counts, reason_test_replies = greedy_eval(reasoning_model, test_items)
reason_prod_counts, reason_prod_replies = greedy_eval(reasoning_model, product_set)

dict_keys(['model_state_dict', 'model_args'])
took 103.3s
type          total  correct  wrong num  format fail  % correct
n+n=?            50       50          0            0     100.0%
n-n=?            50       50          0            0     100.0%
n*n=?            50       50          0            0     100.0%
n/n=?            50       48          2            0      96.0%
x+n=n,x=?        50       50          0            0     100.0%
n+x=n,x=?        50       50          0            0     100.0%
x-n=n,x=?        50       50          0            0     100.0%
n-x=n,x=?        50       50          0            0     100.0%
x*n=n,x=?        50       49          1            0      98.0%
n*x=n,x=?        50       47          3            0      94.0%
x/n=n,x=?        50       50          0            0     100.0%
n/x=n,x=?        50       45          5            0      90.0%
All             600      589         11            0      98.2%
took 541.8s
type          total  correct  wron

## References

- A. Karpathy, *nanoGPT*, https://github.com/karpathy/nanoGPT. `model.py` and `configurator.py` come from this repository and are not modified.
- Course base code, https://github.com/zhouyuan888888/NanoGPT-Math. It provides the pretrained model `sft/gpt.pt`, the character codebook `sft/meta.pkl`, the sample `pos_neg_pairs.json`, and the template notebook: the code of Steps 1–4 (Step 2 with the changes listed there), `compute_logprob`, `pad_or_truncate`, `get_batches`, the skeletons of Steps 7 and 8, and the suggested loss.
- R. Rafailov, A. Sharma, E. Mitchell, S. Ermon, C. D. Manning, C. Finn, "Direct Preference Optimization: Your Language Model is Secretly a Reward Model", NeurIPS 2023, https://arxiv.org/abs/2305.18290.
- N. Lee, K. Sreenivasan, J. D. Lee, K. Lee, D. Papailiopoulos, "Teaching Arithmetic to Small Transformers", ICLR 2024, https://arxiv.org/abs/2307.03381.
- PyTorch documentation for `torch.optim.AdamW`, `torch.optim.lr_scheduler.CosineAnnealingLR` and `torch.nn.utils.clip_grad_norm_`.